## Metric View: IITB Subreddit Metrics

This notebook creates a **metric view** for analytics on r/iitbombay subreddit data.

**Configuration**: Use the widget inputs at the top of this notebook to set your catalog and schema.

**Prerequisites**: Run `01-data-ingestion.ipynb` and `02-data-transformation/` SQL files first.

In [ ]:
# Widget setup - configure your catalog and schema
dbutils.widgets.text("catalog", "dbdemos_vishesh", "Catalog Name")
dbutils.widgets.text("schema", "bharat_bricks", "Schema Name")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
print(f"Creating metric view in: {catalog}.{schema}")

In [ ]:
%sql
CREATE OR REPLACE VIEW ${catalog}.${schema}.iitb_subreddit_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1

  source: >
    SELECT
      p.post_id,
      p.title,
      p.body,
      p.author,
      p.created_at,
      p.score        AS post_score,
      p.upvote_ratio,
      p.num_comments,
      p.flair,
      p.author_flair_text,
      p.is_self,
      p.is_video,
      p.is_original_content,
      p.spoiler,
      p.locked,
      p.domain,
      p.num_crossposts,
      p.subreddit_subscribers,
      p.content_type,
      p.permalink,
      COALESCE(c.total_comment_score, 0)  AS total_comment_score,
      COALESCE(c.avg_comment_score,  0.0) AS avg_comment_score,
      COALESCE(c.max_thread_depth,   0)   AS max_thread_depth,
      COALESCE(c.top_level_comments, 0)   AS top_level_comments,
      COALESCE(c.op_replies,         0)   AS op_replies,
      COALESCE(c.unique_commenters,  0)   AS unique_commenters,
      COALESCE(c.total_awards,       0)   AS total_awards,
      COALESCE(c.edited_comments,    0)   AS edited_comments,
      COALESCE(c.mod_actions,        0)   AS mod_actions
    FROM ${catalog}.${schema}.gold_posts p
    LEFT JOIN (
      SELECT
        post_id,
        SUM(score)                                                    AS total_comment_score,
        AVG(CAST(score AS DOUBLE))                                    AS avg_comment_score,
        MAX(depth)                                                    AS max_thread_depth,
        SUM(CASE WHEN depth = 0 THEN 1 ELSE 0 END)                   AS top_level_comments,
        SUM(CASE WHEN is_submitter THEN 1 ELSE 0 END)                AS op_replies,
        COUNT(DISTINCT author)                                        AS unique_commenters,
        SUM(total_awards_received)                                    AS total_awards,
        SUM(CASE WHEN edited != 'false' THEN 1 ELSE 0 END)           AS edited_comments,
        SUM(CASE WHEN distinguished = 'moderator' THEN 1 ELSE 0 END) AS mod_actions
      FROM ${catalog}.${schema}.gold_comments
      GROUP BY post_id
    ) c ON p.post_id = c.post_id

  comment: >-
    Unified metric view for the r/iitbombay subreddit — IIT Bombay campus life,
    academics, placements, hostel culture, politics, and community discussions.
    Posts are the primary grain, enriched with pre-aggregated comment statistics.
    Use for campus sentiment, engagement trends, content analysis, and community
    health dashboards via AI/BI, Genie, or RAG agents.

  dimensions:
    - name: Post Date
      expr: DATE(created_at)
      display_name: Post Date
      comment: Date (UTC) the post was submitted to r/iitbombay.
      format:
        type: date
        date_format: year_month_day
      synonyms: [date, day, submission date, posted on]

    - name: Post Month
      expr: DATE_TRUNC('MONTH', created_at)
      display_name: Post Month
      comment: Calendar month when the post was submitted.
      format:
        type: date
        date_format: locale_short_month
      synonyms: [month, monthly]

    - name: Post Year
      expr: YEAR(created_at)
      display_name: Year
      comment: Year the post was submitted.
      synonyms: [year, yearly, annual]

    - name: Academic Term
      expr: |-
        CASE
          WHEN MONTH(created_at) BETWEEN 1 AND 4  THEN 'Spring Semester'
          WHEN MONTH(created_at) BETWEEN 5 AND 7  THEN 'Summer Break'
          WHEN MONTH(created_at) BETWEEN 8 AND 12 THEN 'Autumn Semester'
        END
      display_name: Academic Term
      comment: Approximate IIT Bombay academic calendar mapping.
      synonyms: [semester, term, academic period, sem]

    - name: Flair
      expr: COALESCE(flair, 'Untagged')
      display_name: Post Flair
      comment: Subreddit category tag (Question, Acads, Tech, etc.).
      synonyms: [category, topic, tag, post type]

    - name: Content Type
      expr: content_type
      display_name: Content Type
      comment: Media format (text, image, gallery, video, link).
      synonyms: [media type, post format]

    - name: Author
      expr: author
      display_name: Author
      comment: Reddit username of the post author.
      synonyms: [poster, user, username, OP]

    - name: Author Affiliation
      expr: COALESCE(author_flair_text, 'Unknown')
      display_name: Author Affiliation
      comment: Department, hostel, or club badge.
      synonyms: [department, hostel, affiliation, branch]

  measures:
    - name: Total Posts
      expr: COUNT(1)
      display_name: Total Posts
      comment: Total number of posts submitted.
      format: {type: number, decimal_places: {type: exact, places: 0}}
      synonyms: [post count, number of posts]

    - name: Total Comments
      expr: SUM(num_comments)
      display_name: Total Comments
      comment: Sum of comment counts across all posts.
      format: {type: number, decimal_places: {type: exact, places: 0}}
      synonyms: [comment count, replies]

    - name: Avg Post Score
      expr: AVG(post_score)
      display_name: Avg Post Score
      comment: Mean net vote score per post.
      format: {type: number, decimal_places: {type: exact, places: 1}}
      synonyms: [average score, mean score]

    - name: Avg Comments per Post
      expr: AVG(num_comments)
      display_name: Avg Comments per Post
      comment: Mean number of comments per post.
      format: {type: number, decimal_places: {type: exact, places: 1}}
      synonyms: [comments per post, avg replies]

    - name: High Engagement Posts
      expr: SUM(CASE WHEN num_comments >= 20 OR post_score >= 100 THEN 1 ELSE 0 END)
      display_name: High Engagement Posts
      comment: Posts with 20+ comments or 100+ score.
      format: {type: number, decimal_places: {type: exact, places: 0}}
      synonyms: [viral posts, trending posts]

    - name: High Engagement Rate
      expr: SUM(CASE WHEN num_comments >= 20 OR post_score >= 100 THEN 1 ELSE 0 END) / NULLIF(COUNT(1), 0)
      display_name: High Engagement Rate
      comment: Fraction of posts classified as high-engagement.
      format: {type: percentage, decimal_places: {type: exact, places: 1}}
      synonyms: [viral rate]

    - name: Unique Authors
      expr: COUNT(DISTINCT author)
      display_name: Unique Authors
      comment: Number of distinct Reddit users who authored posts.
      format: {type: number, decimal_places: {type: exact, places: 0}}
      synonyms: [unique posters, active posters]
$$